# 53 · Gráficos do relatório de negócios

Gera os 4 gráficos do Customer Segmentation Report (`negocios/`), a partir dos dados brutos e das saídas dos notebooks 39 a 51.

| Gráfico | Pergunta | Arquivo |
|---|---|---|
| Churn mensal | O churn subiu em 2026, e quando? | `saidas/negocios_churn_mensal.png` |
| Churn por plano | A alta está em qual plano? | `saidas/negocios_churn_por_plano.png` |
| Saída × renovação | Quanto da saída de cada segmento já estava decidida? | `saidas/negocios_saida_renovacao.png` |
| Campanha × segmento | Em qual segmento o contato de agosto mudou a saída? | `saidas/negocios_campanha_segmentos.png` |

Regras:
- **Churn mensal:** perdas no mês ÷ ativos no início do mês, sem as 60 contas de teste.
- **Saída por segmento:** foto de 30/06 do notebook 51 (`segmentos_foto_30jun.parquet`). A renovação em 30/06 vem de `eventos_assinatura.csv`, com o horário das lojas convertido de UTC para Brasília.
- **Campanha:** a mesma régua do notebook 51 aplicada em 09/08, véspera do envio, ou seja, antes do tratamento. O desfecho é sair ou desligar a renovação até 30/09. Cada mensagem é comparada com o controle por um teste z de duas proporções.

Ordem de execução: 39 → 41 → 42 → 43 → 50 → 51 → 53.

In [1]:
import sys, warnings
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt, matplotlib.dates as mdates
from pathlib import Path
from scipy.stats import norm
sys.path.insert(0, "../src")
from engajamento import niveis_engajamento
P = Path("../dados/processados"); S = Path("../saidas"); B = Path("../dados/brutos")
CORTE = pd.Timestamp("2026-09-30")
NUM = ["treinos_por_semana", "dias_sem_treinar_log", "duracao_dias_log", "pct_ar_livre", "engajamento_ord"]
ORD = {"possível churn": 0, "baixo": 1, "médio": 2, "alto": 3}
AR = ["corrida", "caminhada", "pedal"]; REGRA = "Inativos em 90 dias (regra)"
SEG = ["Recém-chegados", "Esfriando", REGRA, "Ativos ao ar livre", "Ativos em ambiente fechado"]
LAB = ["Recém-chegados", "Esfriando", "Inativos (90 dias)", "Ativos ar livre", "Ativos fechado"]
MESES = "jan fev mar abr mai jun jul ago set out nov dez".split()
br = lambda x, d=1: f"{x:.{d}f}".replace(".", ",")
plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False, "font.size": 10, "axes.titleweight": "bold", "savefig.dpi": 200})

## 1. Churn mensal e churn por plano

In [2]:
a = pd.read_csv(B / "assinaturas.csv", parse_dates=["inicio", "data_fim"])
a = a[a.plano_atual != "cortesia"]        # 60 contas de teste (e-mail @vitaliza.com.br)
def serie(df):
    out = []
    for m in pd.date_range("2025-01-01", "2026-09-01", freq="MS"):
        e = m + pd.offsets.MonthBegin(1)
        act = df[(df.inicio < m) & (df.data_fim.isna() | (df.data_fim >= m))]
        out.append((m, len(act), int(((act.data_fim >= m) & (act.data_fim < e)).sum())))
    return pd.DataFrame(out, columns=["mes", "ativos", "perdas"]).assign(churn=lambda d: d.perdas / d.ativos * 100)
s = serie(a); sm = serie(a[a.plano_atual == "mensal"]); sa = serie(a[a.plano_atual == "anual"])
m25, m26 = s[s.mes.dt.year == 2025].churn.mean(), s[s.mes.dt.year == 2026].churn.mean()
print(f"média 2025 {m25:.2f}% | média jan-set/26 {m26:.2f}% | abr/26:", s[s.mes == "2026-04-01"][["ativos", "perdas", "churn"]].round(2).to_dict("records"))
fmt = plt.FuncFormatter(lambda v, _: f"{MESES[mdates.num2date(v).month-1]}/{mdates.num2date(v).strftime('%y')}")

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(s.mes, s.churn, color="#2B2D42", lw=2, marker="o", ms=4)
for y, c, l in [(m25, "#8D99AE", "média 2025"), (m26, "#C8553D", "média 2026")]:
    ax.axhline(y, ls="--", lw=1, color=c); ax.text(s.mes.iloc[0], y + .08, f"{l}: {br(y)}%", color=c, fontsize=9)
for d, l in [("2026-03-01", "reajuste do mensal\nR$ 34,90 → 39,90"), ("2026-06-01", "início da pesquisa\nde saída"), ("2026-08-10", "campanha\n10/08")]:
    ax.axvline(pd.Timestamp(d), color="#E9C46A", lw=1.5); ax.text(pd.Timestamp(d) - pd.Timedelta(days=4), 4.55, l, fontsize=7.5, ha="right", va="center")
ax.set_ylim(4, 8.6); ax.set_ylabel("churn mensal (%)"); ax.xaxis.set_major_formatter(fmt)
ax.set_title("O churn subiu em 2026 e teve o pico em abril (8,1%), logo após o reajuste do mensal")
fig.savefig(S / "negocios_churn_mensal.png", bbox_inches="tight", facecolor="white"); plt.close(fig)

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(sm.mes, sm.churn, color="#C8553D", lw=2, marker="o", ms=4, label="mensal")
ax.plot(sa.mes, sa.churn, color="#457B9D", lw=2, marker="o", ms=4, label="anual")
ax.axvspan(pd.Timestamp("2026-03-01"), pd.Timestamp("2026-05-31"), color="#E9C46A", alpha=.25)
v = sm.set_index("mes").churn
ax.text(pd.Timestamp("2026-03-08"), 6.8, f"mar–mai/26:\nmensal {br(v['2026-03-01'])}% →\n{br(v['2026-04-01'])}% → {br(v['2026-05-01'])}%", fontsize=8.5)
ax.set_ylabel("churn mensal (%)"); ax.legend(frameon=False); ax.xaxis.set_major_formatter(fmt)
ax.set_title("A alta de 2026 está no plano mensal: o pico vem logo depois do reajuste de março")
fig.savefig(S / "negocios_churn_por_plano.png", bbox_inches="tight", facecolor="white"); plt.close(fig)
w = lambda d, y: d[d.mes.dt.year == y].perdas.sum() / d[d.mes.dt.year == y].ativos.sum() * 100
print(f"mensal {w(sm,2025):.1f}% → {w(sm,2026):.1f}% | anual {w(sa,2025):.1f}% → {w(sa,2026):.1f}%")

média 2025 5.62% | média jan-set/26 6.62% | abr/26: [{'ativos': 14237, 'perdas': 1154, 'churn': 8.11}]


mensal 8.9% → 10.9% | anual 2.7% → 3.3%


## 2. Renovação na data, pelos eventos

In [3]:
ev = pd.read_csv(B / "eventos_assinatura.csv", dtype=str)
t = pd.to_datetime(ev.data_hora.str.rstrip("Z"), format="ISO8601")
ev["t"] = np.where(ev.data_hora.str.endswith("Z"), t - pd.Timedelta(hours=3), t)   # lojas em UTC → Brasília
ev = ev[ev.evento.isin(["renovacao_automatica_desligada", "renovacao_automatica_religada"]) & ev.plano.notna()]
def desligada_em(d):
    x = ev[ev.t < pd.Timestamp(d) + pd.Timedelta(days=1)].sort_values("t").groupby("user_id").evento.last()
    return x.eq("renovacao_automatica_desligada")

## 3. Saída jul–set por segmento, com e sem quem já tinha desligado a renovação em 30/06

In [4]:
f = pd.read_parquet(P / "segmentos_foto_30jun.parquet")
f["desligada"] = f.user_id.map(desligada_em("2026-06-30")).fillna(False).astype(bool)
g = f.groupby("segmento").agg(n=("user_id", "size"), saida=("saiu_jul_set", "mean"), desligada=("desligada", "mean"))
g["saida_renov_ligada"] = f[~f.desligada].groupby("segmento").saiu_jul_set.mean()
g["pct_das_saidas"] = f[f.saiu_jul_set].groupby("segmento").size() / f.saiu_jul_set.sum()
g["saidas_ja_desligadas"] = f[f.saiu_jul_set].groupby("segmento").desligada.mean()
g = g.loc[SEG]; print((g * [1, 100, 100, 100, 100, 100]).round(1).to_string())
base_lig = f[~f.desligada].saiu_jul_set.mean() * 100
print(f"base: saída {f.saiu_jul_set.mean()*100:.1f}% | com renovação ligada {base_lig:.1f}% | saídas já desligadas {f[f.saiu_jul_set].desligada.mean()*100:.1f}%")

fig, ax = plt.subplots(figsize=(10, 4.3)); x = np.arange(5); wd = .38
tt, ll = g.saida * 100, g.saida_renov_ligada * 100
ax.bar(x - wd/2, tt, wd, color="#8D99AE", label="todos os ativos em 30/06 (número do deck)")
ax.bar(x + wd/2, ll, wd, color="#2B2D42", label="só quem ainda tinha a renovação ligada")
for i in range(5):
    ax.text(x[i] - wd/2, tt.iloc[i] + .6, br(tt.iloc[i]) + "%", ha="center", fontsize=9)
    ax.text(x[i] + wd/2, ll.iloc[i] + .6, br(ll.iloc[i]) + "%", ha="center", fontsize=9, fontweight="bold")
    ax.text(x[i], -4.6, f"{br(g.desligada.iloc[i]*100)}% já tinham\ndesligado em 30/06", ha="center", fontsize=8, color="#C8553D")
ax.set_xticks(x, LAB); ax.set_ylim(-7, 32); ax.axhline(0, color="#999", lw=.8); ax.set_yticks(range(0, 31, 5))
ax.set_ylabel("saíram de jul a set/2026 (%)"); ax.legend(frameon=False, fontsize=9, loc="upper right")
ax.set_title("Tirando quem já tinha desligado a renovação, o maior risco é dos Recém-chegados")
fig.savefig(S / "negocios_saida_renovacao.png", bbox_inches="tight", facecolor="white"); plt.close(fig)

                                n  saida  desligada  saida_renov_ligada  pct_das_saidas  saidas_ja_desligadas
segmento                                                                                                     
Recém-chegados               1681   19.0        5.3                17.8            13.6                  11.0
Esfriando                    3179   26.6       50.5                14.7            36.2                  72.7
Inativos em 90 dias (regra)   753   25.8       30.1                11.8             8.3                  68.0
Ativos ao ar livre           4787   12.6        5.8                10.5            25.7                  21.0
Ativos em ambiente fechado   3659   10.4        3.8                 8.9            16.3                  18.3
base: saída 16.7% | com renovação ligada 11.6% | saídas já desligadas 41.8%


## 4. Campanha de 10/08 por segmento (segmento medido em 09/08, antes do envio)

In [5]:
m = pd.read_parquet(P / "tabela_master.parquet")
ativ = pd.read_parquet(P / "atividades_limpa.parquet", columns=["user_id", "data", "tipo"])
pad = pd.read_csv(S / "segmentacao_oficial_padronizacao.csv").set_index("variavel").loc[NUM]
cen = pd.read_csv(S / "segmentacao_oficial_centros.csv").set_index("segmento")[NUM]
def atribui(d):
    z = (d[NUM].astype(float) - pad.media_elegiveis) / pad.desvio_elegiveis
    cz = (cen - pad.media_elegiveis) / pad.desvio_elegiveis
    return pd.Series(cen.index[((z.to_numpy()[:, None, :] - cz.to_numpy()[None, :, :]) ** 2).sum(axis=2).argmin(axis=1)], index=d.index)
def foto(data):   # mesmo procedimento do notebook 51
    a_ = m[m.ja_assinou & (m.data_inicio_assinatura <= data)]
    a_ = a_[a_.ativa_no_corte | (a_.data_fim > data)]
    f_ = a_[["user_id", "plano", "data_inicio_assinatura", "ativa_no_corte", "data_fim"]].copy().reset_index(drop=True)
    t_ = ativ[ativ.user_id.isin(f_.user_id) & (data - ativ.data).dt.days.between(0, 89)]
    gg = t_.groupby("user_id").agg(treinos_90d=("data", "size"), ar_livre=("tipo", lambda s_: s_.isin(AR).sum()))
    f_ = f_.merge(gg, on="user_id", how="left"); f_["treinos_90d"] = f_.treinos_90d.fillna(0).astype(int)
    f_["treinos_por_semana"] = (f_.treinos_90d / (90 / 7)).round(2)
    f_["pct_ar_livre"] = f_.ar_livre / f_.treinos_90d.where(f_.treinos_90d > 0) * 100
    f_["duracao_dias_log"] = np.log1p((data - f_.data_inicio_assinatura).dt.days).round(4)
    en = niveis_engajamento(ativ, f_.user_id, data, inicio=f_.set_index("user_id").data_inicio_assinatura)
    f_ = f_.merge(en[["user_id", "nivel", "dias_sem_treinar"]], on="user_id", how="left")
    f_["engajamento_ord"] = f_.nivel.str[3:].map(ORD)
    f_["dias_sem_treinar_log"] = np.log1p(f_.dias_sem_treinar.astype(float)).round(4)
    f_["segmento"] = REGRA; el = f_.treinos_90d > 0; f_.loc[el, "segmento"] = atribui(f_[el])
    return f_
chk = foto(pd.Timestamp("2026-06-30")).merge(f[["user_id", "segmento"]], on="user_id", suffixes=("", "_51"))
print("conferência com o notebook 51 (30/06):", f"{(chk.segmento == chk.segmento_51).mean():.2%} iguais")
v = foto(pd.Timestamp("2026-08-09"))
v["desl_antes"] = v.user_id.map(desligada_em("2026-08-09")).fillna(False).astype(bool)
v["y"] = v.data_fim.between(pd.Timestamp("2026-08-10"), CORTE) | v.user_id.map(desligada_em(CORTE)).fillna(False).astype(bool)
camp = pd.read_parquet(P / "campanha_limpa.parquet")[["user_id", "grupo"]]
x = v.merge(camp, on="user_id"); print("sorteados com segmento na véspera:", len(x), "de", len(camp))
def p2(a1, n1, a2, n2):
    p = (a1 + a2) / (n1 + n2); z = (a1 / n1 - a2 / n2) / np.sqrt(p * (1 - p) * (1 / n1 + 1 / n2)); return 2 * norm.sf(abs(z))
rows = []
for rec, d in [("todos", x), ("renovação ligada", x[~x.desl_antes])]:
    for s_ in SEG:
        gg = d[d.segmento == s_]; ct = gg[gg.grupo == "controle"]
        for gr in ["controle", "desafio", "cupom", "conteudo"]:
            h = gg[gg.grupo == gr]
            rows.append({"recorte": rec, "segmento": s_, "grupo": gr, "n": len(h), "saiu_ou_desligou_%": h.y.mean() * 100,
                         "p": np.nan if gr == "controle" else p2(h.y.sum(), len(h), ct.y.sum(), len(ct))})
C = pd.DataFrame(rows); C["bonferroni_15"] = C.p < 0.05 / 15
C.round(4).to_csv(S / "negocios_campanha_por_segmento.csv", index=False, encoding="utf-8-sig")
print(C[C.recorte == "renovação ligada"].round(3).to_string(index=False))

conferência com o notebook 51 (30/06): 100.00% iguais


sorteados com segmento na véspera: 14072 de 14120
         recorte                    segmento    grupo    n  saiu_ou_desligou_%     p  bonferroni_15
renovação ligada              Recém-chegados controle  394              18.528   NaN          False
renovação ligada              Recém-chegados  desafio  391               8.184 0.000           True
renovação ligada              Recém-chegados    cupom  406              10.345 0.001           True
renovação ligada              Recém-chegados conteudo  421              14.014 0.080          False
renovação ligada                   Esfriando controle  378              15.344   NaN          False
renovação ligada                   Esfriando  desafio  397              18.388 0.258          False
renovação ligada                   Esfriando    cupom  335              17.612 0.414          False
renovação ligada                   Esfriando conteudo  334              16.467 0.682          False
renovação ligada Inativos em 90 dias (regra) contr

In [6]:
d = C[C.recorte == "renovação ligada"]; fig, axs = plt.subplots(1, 5, figsize=(14, 4.2), sharey=True)
col = {"controle": "#8D99AE", "desafio": "#2A9D8F", "cupom": "#E9C46A", "conteudo": "#457B9D"}
nm = {"controle": "controle", "desafio": "desafio", "cupom": "cupom", "conteudo": "conteúdo"}
for ax, s_, lb in zip(axs, SEG, LAB):
    gg = d[d.segmento == s_]; xx = np.arange(len(gg))
    ax.bar(xx, gg["saiu_ou_desligou_%"], color=[col[k] for k in gg.grupo], width=.75)
    for i, (val, p) in enumerate(zip(gg["saiu_ou_desligou_%"], gg.p)):
        star = "" if pd.isna(p) else ("*" if p < 0.05 else "")
        ax.text(i, val + .8, f"{br(val)}%{star}", ha="center", fontsize=8.5, fontweight="bold" if star else "normal")
    ax.set_xticks(xx, [nm[k] for k in gg.grupo], rotation=35, ha="right", fontsize=8.5); ax.set_title(lb, fontsize=10.5)
axs[0].set_ylabel("saiu ou desligou a renovação\naté 30/09 (%)"); axs[0].set_ylim(0, 42)
fig.suptitle("Campanha de 10/08, só quem tinha renovação ligada: contato ajudou Recém-chegados e Ativos, piorou os Inativos", fontweight="bold", fontsize=11, y=1.0)
fig.text(.5, -.06, "* diferença significativa contra o controle (p < 0,05). Segmento oficial medido em 09/08, véspera do envio. Grupos de 117 a 1.157 pessoas.", ha="center", fontsize=8.5, color="#555")
fig.tight_layout(); fig.savefig(S / "negocios_campanha_segmentos.png", bbox_inches="tight", facecolor="white"); plt.close(fig)
print("gráficos gravados em saidas/: negocios_churn_mensal.png, negocios_churn_por_plano.png, negocios_saida_renovacao.png, negocios_campanha_segmentos.png")

gráficos gravados em saidas/: negocios_churn_mensal.png, negocios_churn_por_plano.png, negocios_saida_renovacao.png, negocios_campanha_segmentos.png
